<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Multi-Tool Agents
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Mit mehreren Tools wird **Planen und Handeln** zur echten Aufgaben — der Meeting- & Briefing-Agent muss pro Schritt wählen, welches Tool zum aktuellen Teilproblem passt, und dieses Werkzeug gezielt ausführen. Jedes Tool bleibt dabei sein eigener Agenten-Vertrag mit klar abgegrenzter Verantwortung (siehe Designregeln unten).

**Praxisanker:** Mara Vogt erwartet, dass der Agent zwischen Anfrage-Einordnung, Quellensuche, Quellenhinweis und Risikoprüfung passend auswählt.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
# Globales Tracing aus: Nur der dedizierte Multi-Tool-Referenzlauf wird erfasst.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"]    = "m05-multi-tool-agents"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---

In *Erste Agenten mit LangChain* wurde ein erster Agent mit **2 Briefing-Tools** gebaut. Dieses Modul geht einen Schritt weiter:
**4 fachliche Tools plus Reflexion**, komplexere Anfragen und ein genauer Blick darauf, wie das LLM entscheidet, welches Tool es einsetzt.

| Merkmal | Einfacher Agent | Multi-Tool Agent |
|---|---|---|
| **Anzahl Tools** | 2 | 4 + Reflexion |
| **Aufgabentyp** | Einzel-Tool-Anfragen | Multi-Step-Anfragen |
| **Lernfokus** | Wie funktioniert ein Agent? | Warum wählt er Tool X? |
| **Debugging** | Direkte Ausgabe | LangSmith Trace |

**Einordnung im Lernpfad:**

Hier steht die Entscheidung zwischen mehreren Werkzeugen im Mittelpunkt. Eine Chain eignet sich für fest komponierte Teilabläufe; die Tool-Auswahl übernimmt in diesem Beispiel der Agent.


In [ ]:
from langchain.chat_models import init_chat_model

from langchain_core.messages import HumanMessage, AIMessage
# ── Konfigurationskonstante ────────────────────────────────────────────────
MAX_TOOL_CALLS = 10  # Maximale Tool-Aufrufe pro Agent-Lauf (recursion_limit)

llm = init_chat_model(BASELINE)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>


diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph EINFACH["Einfacher Agent"]
        direction LR
        F1[Anfrage] --> AG1[Agent]
        AG1 <-->|Anfrage / Tool-Call| LLM1[LLM]
        AG1 -->|führt ein Tool aus| T1A[Tool 1 oder 2]
        T1A -->|Ergebnis| AG1
        AG1 --> ANT1[Antwort]
    end
    subgraph MULTI["Multi-Tool Agent"]
        direction LR
        F2[Briefing-Anfrage] --> AG2[Agent]
        AG2 <-->|Anfrage / Tool-Call| LLM2[LLM]
        AG2 -->|Schritt 1| T2A[Briefing-Signal]
        T2A -->|Ergebnis 1| AG2
        AG2 -->|Schritt 2| T2B[Dokumentenpassung]
        T2B -->|Ergebnis 2| AG2
        AG2 -->|Schritt 3| T2C[Quellenhinweis]
        T2C -->|Ergebnis 3| AG2
        AG2 --> ANT2[Antwort]
    end
"""
mermaid(diagram, width=800)


# 2 | Agent mit 4 Tools
---


Jedes Tool übernimmt **eine klar abgegrenzte Aufgabe** im Briefing-Agent-Workflow.
So kann der Agent mit einer einzigen Anfrage mehrere fachliche Prüfungen koordinieren.

| Tool | Domäne | Aufgaben |
|---|---|---|
| `anfrage_einordnen` | Anfrageanalyse | Fachbegriffe und Fragetyp erkennen |
| `thema_einordnen` | Dokumentenpassung | Kursnähe grob prüfen |
| `quellenhinweis_erstellen` | Quellenbindung | Beleganforderung formulieren |
| `risiko_einschaetzen` | Qualität | Risiko einer unbelegten Antwort bewerten |




**Designregeln für gute Tools:**
1. **Eine Verantwortung** - jedes Tool löst genau ein Problem
2. **Präziser Docstring** - das LLM kennt das Tool nur über seine Beschreibung
3. **Sprechende Parameternamen** - `frage` statt `x`, `thema` statt `y`
4. **Klare Rückgabe** - Fehlermeldungen als String, kein `raise`


In [ ]:
#@markdown   <p><font size="4" color='green'>  Tool-Übersicht</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    AGENT["Agent
gpt-5.4-mini"]
    T1["anfrage_einordnen
Anfrageanalyse"]
    T2["thema_einordnen
Dokumentenpassung"]
    T3["quellenhinweis_erstellen
Quellenbindung"]
    T4["risiko_einschaetzen
Qualität"]

    AGENT --> T1
    AGENT --> T2
    AGENT --> T3
    AGENT --> T4
"""
mermaid(diagram, width=700)


In [ ]:
from langchain_core.tools import tool

@tool
def anfrage_einordnen(text: str) -> str:
    """Extrahiert zentrale Briefing-Signale aus einer Anfrage oder einem kurzen Fachtext.
    text: Anfrage oder Fachtext, aus dem Signale wie Entscheidungen, Risiken, offene Punkte, Action Items, Fristen oder Verantwortliche erkannt werden."""
    briefing_signals = ["entscheidung", "risiko", "offener punkt", "action item", "frist", "verantwortlich", "quelle"]
    treffer = [begriff for begriff in briefing_signals if begriff in text.lower()]
    return ", ".join(treffer) if treffer else "Keine klaren Briefing-Signale gefunden."

@tool
def thema_einordnen(thema: str) -> str:
    """Schätzt ein, ob ein Thema zu den verfügbaren Meeting- und Projektdokumenten passt.
    thema: Begriff, Frage oder Kurzbeschreibung des zu prüfenden Themas."""
    briefing_relevant = ["meeting", "briefing", "entscheidung", "risiko", "aktion", "frist", "projekt"]
    if any(wort in thema.lower() for wort in briefing_relevant):
        return "Thema passt voraussichtlich zu den verfügbaren Meeting- und Projektdokumenten."
    return "Keine sichere Dokumentenpassung erkennbar. Antwort als unsicher markieren."

@tool
def quellenhinweis_erstellen(thema: str, aussage: str) -> str:
    """Erstellt einen Hinweis, wie eine Briefing-Aussage belegt werden sollte.
    thema: Thema oder Frage der Antwort.
    aussage: Kernaussage, die mit Quelldokument, Abschnitt oder Dokument-ID belegt werden soll."""
    return f"Für '{thema}' sollte die Aussage '{aussage}' mit Quelldokument, Abschnitt oder Dokument-ID belegt werden."

@tool
def risiko_einschaetzen(frage: str, hat_quellen: bool) -> str:
    """Bewertet das Risiko einer Briefing-Antwort ohne ausreichende Evidenz.
    frage: Nutzerfrage oder fachliche Teilfrage.
    hat_quellen: True, wenn Quellen oder Kontextpassagen vorliegen; sonst False."""
    if hat_quellen:
        return "Risiko niedrig: Quellenbindung ist vorhanden."
    if any(wort in frage.lower() for wort in ["entscheidung", "risiko", "frist", "verantwortlich", "offen"]):
        return "Risiko hoch: Ohne Quellen kann die Antwort spekulativ wirken."
    return "Risiko mittel: kurze Erklärung möglich, aber Quellenhinweis wäre besser."

@tool
def record_briefing_reflection_demo(reflection: str) -> str:
    """Nur Demo-Helfer: hält eine Reflexionsnotiz fest; kein Produktiv-Tool im Agentenpfad.

    Nutze dieses Tool nach einem Tool-Ergebnis:
    - Ist die Teilfrage vollständig beantwortet?
    - Welches Tool ist als nächstes sinnvoll?
    - Reicht der Kontext für eine belegbare Antwort?

    Args:
        reflection: Analyse des aktuellen Stands und geplanter nächster Schritt
    """
    return f"Reflexion notiert: {reflection}"

mprint("✅ 4 Tools definiert:")
mprint("   anfrage_einordnen · thema_einordnen · quellenhinweis_erstellen")
mprint("   risiko_einschaetzen")


In [ ]:
from langchain.agents import create_agent

tools = [anfrage_einordnen, thema_einordnen, quellenhinweis_erstellen, risiko_einschaetzen]
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m05_multi_tool_system_prompt.md", mode="S")

agent = create_agent(
    model=WORKER,
    tools=tools,
    system_prompt=system_prompt,
)

antwort = agent.invoke({"messages": [HumanMessage(content="Welche Briefing-Signale enthält die Frage: Warum verbessert RAG die Antwortqualität?")]}, config={"run_name": "m05_kap2_agent_antwort"})
mprint("## Erster Test")
mprint(antwort["messages"][-1].content)

# 3 | Komplexe Multi-Step Aufgaben
---



Bei **Multi-Step-Aufgaben** entscheidet das LLM schrittweise, welches Tool als nächstes aufgerufen wird. Ein Agent oder ein Framework steuert dabei die Ausführungsschleife, übergibt Zwischenergebnisse als Kontext und sorgt dafür, dass der nächste Tool-Call erst nach Vorliegen des vorherigen Ergebnisses erfolgt.

> *Das LLM entscheidet lokal - der Agent organisiert global.*

**Typische Multi-Step-Muster:**

| Muster | Beispiel |
|---|---|
| **Parallel**: unabhängige Tools | Briefing-Signale UND Dokumentenpassung prüfen |
| **Sequenziell**: Ergebnis 1 -> Input 2 | Dokumentenpassung, dann Quellenhinweis formulieren |
| **Bedingt**: nur wenn nötig | Risiko nur bewerten, wenn Quellen fehlen |

> **Beobachtung:** Im nächsten Beispiel zählen, wie viele Tool-Calls der Agent benötigt.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Ablauf einer Multi-Step Anfrage</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    participant User
    participant Agent
    participant LLM as LLM (gpt-5.4-mini)
    participant S as anfrage_einordnen
    participant K as thema_einordnen
    participant Q as quellenhinweis_erstellen

    User->>Agent: "Prüfe RAG-Frage, Dokumentenpassung und Quellenbedarf"
    Agent->>LLM: Nutzeranfrage + Tool-Schemas

    loop Solange das LLM Tools aufruft
        LLM-->>Agent: Tool-Call: anfrage_einordnen
        Note over Agent,Q: Der Agent führt die Tools aus,<br/>nicht das LLM
        Agent->>S: text=Anfrage
        S-->>Agent: "rag, retrieval, evaluation"
        Agent->>LLM: Tool-Ergebnis: Briefing-Signale

        LLM-->>Agent: Tool-Call: thema_einordnen
        Agent->>K: thema=RAG Evaluation
        K-->>Agent: "passt voraussichtlich"
        Agent->>LLM: Tool-Ergebnis: Dokumentenpassung

        LLM-->>Agent: Tool-Call: quellenhinweis_erstellen
        Agent->>Q: thema=RAG, aussage=Zuverlässigkeit verbessern
        Q-->>Agent: Quellenhinweis
        Agent->>LLM: Tool-Ergebnis: Quellenhinweis
    end

    LLM-->>Agent: Finale Antwort
    Agent-->>User: Antwort mit Tool-Ergebnissen
"""
mermaid(diagram, width=1100)


In [ ]:
komplexe_anfrage = (
    "Prüfe für den Meeting-Briefing-Agent: "
    "1. Welche Briefing-Signale enthält die Frage 'Warum verbessert RAG die Zuverlässigkeit?' "
    "2. Passt das Thema zu den verfügbaren Meeting- und Projektdokumenten? "
    "3. Welche Quellenbindung wäre für die Antwort sinnvoll?"
)

ergebnis_multi = agent.invoke({"messages": [HumanMessage(content=komplexe_anfrage)]}, config={"run_name": "m05_kap3_ergebnis_multi"})

mprint("## Multi-Step Ergebnis")
mprint(ergebnis_multi["messages"][-1].content)

In [ ]:
mprint("## Tool-Calls in dieser Anfrage")
schritt = 1
for msg in ergebnis_multi["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            mprint(f"**Schritt {schritt}:** `{tc['name']}` mit Args `{tc['args']}`")
            schritt += 1
if schritt == 1:
    mprint("*Kein Tool-Call – direkte LLM-Antwort*")

# 4 | Agent-Debugging mit LangSmith
---



Bei Multi-Tool Agenten wird LangSmith besonders wertvoll: Der vollständige ReAct-Zyklus
 ist im Trace sichtbar - **für jeden einzelnen Tool-Call**.

**Was im Multi-Tool Trace sichtbar wird:**

| Trace-Element | Was es zeigt |
|---|---|
| **LLM-Call (Think)** | Wie das LLM die Anfrage und bisherigen Ergebnisse verarbeitet |
| **Tool-Call (Act)** | Welches Tool mit welchen Argumenten aufgerufen wurde |
| **Tool-Response (Observe)** | Was das Tool zurückgegeben hat |
| **Iterationen** | Wie oft der TAO-Zyklus wiederholt wurde |
| **Token-Verbrauch** | Gesamt und pro Schritt |



<p><font color='darkblue' size="4">💡 <b>Tipp</b></font></p>

LangSmith parallel öffnen und beobachten, wie der Agent durch die Zwischenschritte navigiert. Der Trace zeigt mehr als die finale Antwort.


In [ ]:
run_cfg = {
    "run_name": "m05_kap4_multitooltrace",
    "tags":     ["m05", "multi-tool", "langsmith", "research"]
}

trace_anfrage = (
    "Analysiere die Frage 'Wie vergleicht man RAG und Fine-Tuning?' "
    "Prüfe Dokumentenpassung, Quellenbedarf und Risiko ohne Quellen."
)

from langsmith import tracing_context

with tracing_context(enabled=True):
    trace_ergebnis = agent.invoke({"messages": [HumanMessage(content=trace_anfrage)]}, config=run_cfg)

mprint("## Trace-Ergebnis")
mprint(trace_ergebnis["messages"][-1].content)

In [ ]:
import time
from langsmith import Client

time.sleep(3)  # Trace-Upload in LangSmith abwarten
runs = list(Client().list_runs(
    project_name="m05-multi-tool-agents",
    filter=f'eq(name, "{run_cfg["run_name"]}")',
    is_root=True,
    limit=5,
    select=["name", "start_time", "prompt_tokens", "completion_tokens", "total_tokens",
            "prompt_cost", "completion_cost", "total_cost"],
))

if not runs:
    print("Kein Trace in LangSmith gefunden - Upload evtl. noch nicht abgeschlossen, Zelle erneut ausführen.")
else:
    run = max(runs, key=lambda r: r.start_time)

    def usd(wert):
        return f"${wert:.5f}" if wert is not None else "n/a"

    print(f"Token-Verbrauch und Kosten laut LangSmith (Run: {run.name})")
    print("-" * 40)
    print(f"Input : {run.prompt_tokens or 0:>6} Tokens | {usd(run.prompt_cost)}")
    print(f"Output: {run.completion_tokens or 0:>6} Tokens | {usd(run.completion_cost)}")
    print("-" * 40)
    print(f"Total : {run.total_tokens or 0:>6} Tokens | {usd(run.total_cost)}")
    if run.total_cost is None:
        print("Hinweis: LangSmith hat für dieses Modell keinen Preis hinterlegt (in den LangSmith-Einstellungen pflegen).")



<p><font color='black' size="5">
Token- und Kostenüberblick
</font></p>

Jeder Agent-Aufruf verbraucht Tokens - bei Multi-Step-Anfragen summiert sich das über mehrere LLM-Calls hinweg. LangSmith summiert Tokens und Kosten über alle LLM-Calls eines Runs.

> **LangSmith zeigt Kosten pro Run** - dieselben Werte (`total_tokens`, `total_cost`) lassen sich per API direkt im Notebook abrufen.

# 5 | Warum wählt der Agent Tool X?
---



Das LLM kennt die Tools **ausschließlich über ihre Docstrings**.
Die Beschreibung ist der einzige Kompass für die Tool-Auswahl.

**Was bei der Tool-Auswahl passiert:**
1. Das LLM liest alle Tool-Beschreibungen
2. Es wählt das Tool, dessen Beschreibung am besten zur Anfrage passt
3. Es generiert die Argumente passend zu den Parameternamen

> **Konsequenz:** Ein vager Docstring führt zu falschen Tool-Calls oder dazu,
> dass der Agent ein Tool gar nicht aufruft, obwohl es die richtige Antwort hätte.

Im folgenden Experiment werden zwei Varianten desselben Tools verglichen:
eine mit **vager** und eine mit **präziser** Beschreibung.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Tool-Auswahlprozess des LLM</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    FRAGE["Anfrage: 'Welche Quellenbindung braucht eine RAG-Antwort?'"]
    LLM["LLM vergleicht
Anfrage mit Tool-Docstrings"]
    T1["quellenhinweis_erstellen
passend: Beleganforderung"]
    T2["anfrage_einordnen
nur Begriffe"]
    T3["thema_einordnen
nur Dokumentenpassung"]
    T4["risiko_einschaetzen
nur Risiko"]
    AUSWAHL["Tool-Call-Anforderung:
quellenhinweis_erstellen"]

    FRAGE --> LLM
    LLM -->|bewertet| T1 & T2 & T3 & T4
    T1 --> AUSWAHL

    style T1 fill:#d5e8d4
    style T2 fill:#f8cecc
    style T3 fill:#f8cecc
    style T4 fill:#f8cecc
    style AUSWAHL fill:#dae8fc
"""
mermaid(diagram, width=850)


In [ ]:
@tool
def briefing_tool_vage(x: str) -> str:
    """Prüft etwas."""
    return f"Unklare Prüfung für: {x}"

@tool
def quellenhinweis_erstellen_klar(thema: str, aussage: str) -> str:
    """Erstellt einen Quellenhinweis für eine Briefing-Antwort.
    thema: Fachthema oder Frage, zu der eine Antwort belegt werden soll.
    aussage: Kernaussage, die mit Quelldokument, Abschnitt oder Dokument-ID abgesichert werden soll."""
    return f"Beleg nötig: '{aussage}' zum Thema '{thema}' mit Quelldokument, Abschnitt oder Dokument-ID absichern."

In [ ]:
assistent_prompt = "Rolle: Meeting-Briefing-Agent. Antworte auf Deutsch."
agent_vage = create_agent(model=WORKER, tools=[briefing_tool_vage], system_prompt=assistent_prompt)
agent_klar = create_agent(model=WORKER, tools=[quellenhinweis_erstellen_klar], system_prompt=assistent_prompt)

In [ ]:
testfrage = "Welche Quellenbindung braucht die Aussage, dass RAG die Zuverlässigkeit verbessert?"

r_vage = agent_vage.invoke({"messages": [HumanMessage(content=testfrage)]}, config={"run_name": "m05_kap5_r_vage"})
r_klar = agent_klar.invoke({"messages": [HumanMessage(content=testfrage)]}, config={"run_name": "m05_kap5_r_klar"})

mprint("## Ergebnis: Vage Beschreibung (`'Prüft etwas.'`)")
mprint(r_vage["messages"][-1].content)
mprint("## Ergebnis: Präzise Beschreibung")
mprint(r_klar["messages"][-1].content)

# 6 | Error Handling & Fallbacks
---



Wenn ein Tool einen Fehler wirft (`raise`), bricht der Agent in der Regel ab.
Besser: Das Tool **gibt einen Fehler als String zurück** - dann kann das LLM
die Fehlermeldung lesen, erklären und um eine Korrektur bitten.

| Strategie | Beschreibung | Empfehlung |
|---|---|---|
| `raise Exception` | Tool wirft Fehler | Agent bricht ab |
| `return "Fehler: ..."` | Tool gibt Fehlermeldung zurück | Agent erklärt |
| Validierung vor Logik | Werte prüfen, bevor verarbeitet wird | frühzeitig fangen |

**Typische Fehlerquellen:**
- Leere Anfrage
- Thema ohne passende Meeting- oder Projektdokumente
- Quellenhinweis ohne konkrete Aussage
- Boolean-Parameter unklar formuliert


In [ ]:
@tool
def safe_briefing_topic_fit_check(thema: str) -> str:
    """Prüft robust die Themenpassung zu den verfügbaren Meeting- und Projektdokumenten und gibt Fehlermeldungen als String zurück.
    thema: Fachthema oder Frage; darf nicht leer sein."""
    if not thema or not thema.strip():
        return "Fehler: Thema fehlt. Bitte eine konkrete Briefing-Frage oder ein Fachthema angeben."
    return thema_einordnen.invoke({"thema": thema})

In [ ]:
robust_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m05_robust_research_system_prompt.md", mode="S")
agent_robust = create_agent(
    model=WORKER,
    tools=[safe_briefing_topic_fit_check],
    system_prompt=robust_prompt,
)

r_fehler = agent_robust.invoke({"messages": [HumanMessage(content="Prüfe die Dokumentenpassung ohne Thema.")]}, config={"run_name": "m05_kap6_r_fehler"})
mprint("## Fehlerfall: fehlendes Thema")
mprint(r_fehler["messages"][-1].content)

In [ ]:
# Kombination: Nach Fehler direkt weiter mit gültiger Anfrage
r_kombi = agent_robust.invoke({
    "messages": [
        HumanMessage(content="Prüfe die Dokumentenpassung ohne Thema."),
        AIMessage(content=r_fehler["messages"][-1].content),
        HumanMessage(content="Ok, dann prüfe bitte RAG Evaluation als Thema."),
    ]
}, config={"run_name": "m05_kap6_r_kombi"})
mprint("## Robuste Fortsetzung nach Fehler")
mprint(r_kombi["messages"][-1].content)

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m05-multi-tool-agents", limit=3, show_steps=True)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Eigener Multi-Tool Agent
</font></p>

Einen eigenen Agenten für den Meeting-Briefing-Agent bauen - mit mindestens 3 Tools aus unterschiedlichen Aufgabenbereichen.


**Grundlagen**
- Ein Agent mit 3 Briefing-Tools läuft.
- Mindestens 1 Anfrage führt zu einem Tool-Aufruf.


In [ ]:
# Grundlagen: Agent mit 3 Tools, Briefing-Anfrage testen
meine_tools = [anfrage_einordnen, thema_einordnen, quellenhinweis_erstellen]

# ...


**Aufbau**
- Mindestens **3 selbst definierte Briefing-Tools** aus verschiedenen Aufgabenbereichen.
- Präzise Docstrings mit deutschen Parameterbeschreibungen.
- Einen passenden System-Prompt für die Rolle des Agenten.
- Mindestens **2 Multi-Step Anfragen** testen.
- **Fehlerbehandlung** in mindestens einem Tool: Rückgabe statt `raise`.


In [ ]:
# Aufbau: Präzise Docstrings + Multi-Step-Anfragen
for tool_obj in meine_tools:
    assert len(tool_obj.description) >= 20, f'{tool_obj.name}: Docstring zu kurz'
    print(f'{tool_obj.name}: {len(tool_obj.description)} Zeichen')

# ...


**Vertiefung**
- Das Beschreibungs-Experiment durchführen: absichtlich einen Docstring verschlechtern und beobachten, wie sich das Verhalten des Agenten verändert.
- LangSmith während einer Multi-Step Anfrage öffnen und die TAO-Zyklen zählen.
- Ein 4. Tool hinzufügen und Anfragen testen, die alle 4 Tools benötigen.
- Den Run mit eigenem `run_name` und eigenen `tags` im `config` ausführen und in LangSmith unter diesem Namen wiederfinden.


In [ ]:
# Vertiefung: Docstring-Experiment
_docstring_beobachtung = (
    "Mit vagen Docstrings wählt der Agent Tools unsicherer, weil Einsatzbedingungen fehlen. "
    "Mit präzisen Docstrings verbessert sich die Tool-Auswahl, weil Parameter, Kontext und Ziel klarer beschrieben sind."
)

# ...

# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Weitere Tools](https://ralf-42.github.io/Agenten/05-frameworks/weitere-tools.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [Einsteiger LangChain](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langchain.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
